# Семантический матчинг с помощью библиотеки sentence_transformers

В этом тюториале мы научимся семантическому поиску и ранжированию документов с помощью библиотеки _sentence_transformers_.

Документация библиотеки: https://www.sbert.net/

Ее можно поставить в виртуальное окружение командой `pip install sentence-transformers`

Импортируем модули которые нам понадобятся впоследствии:

In [ ]:
import logging
from timeit import default_timer as timer

import numpy as np
import torch
import sentence_transformers as sbert

Подготовим логгер:

In [ ]:
logging.basicConfig(level=logging.INFO)

## Данные

Будем работать с мини-сэмплом из пассажной части стандарного поискового датасета _MS MARCO_.

MS MARCO содержит:
- большую коллекцию документов, по которым можно искать
- реальные поисковые запросы из логов поисковика bing
- информацию о том, какие документы релевантны данному запросу

Подробнее про _MS MARCO_: https://microsoft.github.io/msmarco/

Возьмем несколько случайных запросов:

In [ ]:
queries = [
    "how long is jupiter day",
    "what are isotopes of hydrogen",
]
queries

И несколько документов.

Выберем документы так, чтобы в нашем сэмпле были как релевантные запросам, так и нерелевантные, но "сложные": или содержащие ключевые слова, или просто близкие по тематике.

In [ ]:
docs = [
    # About Jupiter
    "A day on Jupiter, also known as the sidereal rotation period, lasts 9.92496 hours.", # relevant
    "Voyager 2 arrived at Jupiter in 1979 two years after launching from Cape Canaveral",
    "It's hard to explain what the atmosphere of Jupiter is, since the planet is 90% hydrogen and 10% helium.",
    "The gas giant Jupiter has 63 known moons.",
    "Jupiter, the solar system s largest planet, takes 4,331.572 days to orbit around the sun",
    "Jupiter orbits the Sun every 11.86 Earth years (or 4,332 days)",
    "Given the fact that it is the largest planet in the Solar System, one would expect that a day on Jupiter would last a long time.",
    "Io takes 1.77 Earth days to orbit Jupiter.",

    # About hydrogen
    "Hydrogen has three isotopes: hydrogen-1 (protium), hydrogen-2 (deuterium) and hydrogen-3 (tritium).", # relevant
    "A deuterium atom contains one proton, one neutron, and one electron.",
    "The fusion of deuterium with tritium creates helium-4, frees a remaining neutron, and releases energy.",
    "These different versions of hydrogen are called isotopes.",
    "Hydrogen is the only element where the isotopes are given specific names.",
    "The 2 H (or hydrogen-2) isotope is usually called deuterium",
    "Tritium (hydrogen-3 or 3H) is a weakly radioactive isotope of the element hydrogen",
    "All elements have at least some radioactive isotopes.",
    
    # Misc
    "The official language in Bangladesh is Bengali, though English is widely spoken.",
    "Grasshoppers are medium to large insects.",
    "Debian and Ubuntu are the most influential Linux distributions ever.",
    "Мoscow is the largest city in Russia with a population of 12.1 million people",
]

docs

## Семантический поиск

Начнем с того, что попробуем поискать наши запросы в нашей коллекции документов.

Для этого нам надо:
- заэмбеддить все документы
- заэмбеддить запросы
- найти к каждому запросу ближайшие документы по косинусному расстонию

Для простоты мы не будем использовать какой-либо векторный индекс (типа _faiss_), и вместо приближенного поиска ближайших соседей (ANN) будем делать полный KNN.

Подготовим устройство:

In [ ]:
device = torch.device("cuda:0")
logging.info(f"created device {device}")

Загрузим модели семантического поиска/матчинга.

Таких моделей существует множества, они могу отличаться:
- по качеству
- размеру и скорости
- поддержке разных языков и т.п.

Для выбора моделей рекомендую опираться на бенчмарки, напр. на MTEB:
- тут можно найти описание и ссылку на лидерборд: https://github.com/embeddings-benchmark/mteb
- немного про сам бенчмарк: https://huggingface.co/blog/mteb

Обычно, для русского языка в 2026 г. используют какую-либо модель из:
- семейства моделей E5 от Microsoft, напр. _multilingual-e5-large_
- новых Qwen3-Embedding, напр. _Qwen3-Embedding-8B_
- моделей BGE, напр. _bge-m3_
- моделей от Сбера, напр. _FRIDA_

Мы с вами дальше будем работать с английским языком, поэтому нам подойдут более легкие модели из семейства _MiniLM_.

In [ ]:
# Which model?
model_name = "sentence-transformers/all-MiniLM-L6-v2"
#model_name = "intfloat/multilingual-e5-base"
#model_name = "Qwen/Qwen3-Embedding-0.6B" # requires prompt!

# Model params
model_kwargs = {}
model_kwargs['dtype'] = torch.float16

# Load model
model = sbert.SentenceTransformer(model_name, device=str(device), trust_remote_code=True, model_kwargs=model_kwargs)
num_params = sum(t.numel() for t in model.parameters())
logging.info(f"loaded model: model_name = {model_name} num_params = {num_params}")

Убедимся, что модель загрузилась на GPU с помощью утилиты _nvidia-smi_:

In [ ]:
!nvidia-smi

Заэмбеддим запросы:

In [ ]:
start = timer()
query_embeds = model.encode(queries)
logging.info(f"embedded queries: query_embeds.shape = {query_embeds.shape} elapsed = {timer() - start:.3f}")

И, аналогично, заэмбеддим документы:

In [ ]:
start = timer()
doc_embeds = model.encode(docs)
logging.info(f"embedded docs: doc_embeds.shape = {doc_embeds.shape} elapsed = {timer() - start:.3f}")

Посчитаем скоры близости, как правило это просто косинус между векторами-эмбеддингами:

In [ ]:
scores = model.similarity(query_embeds, doc_embeds).cpu().numpy()
print(scores)
logging.info(f"computed similarities: scores.shape = {scores.shape}")

Напишем функцию, которая ранжирует документы по запросу, и возвращает ранжированный список пар документ-скор:

In [ ]:
def rank_docs(docs, query_scores):
    ranked_doc_ids = np.argsort(query_scores)[::-1]
    ranked_docs = [docs[i] for i in ranked_doc_ids]
    doc_scores = query_scores[ranked_doc_ids].tolist()
    docs_with_scores = list(zip(ranked_docs, doc_scores))
    return docs_with_scores

Применим к первому запросу про Юпитер:

In [ ]:
print(f"Query: {queries[0]}")
rank_docs(docs, scores[0])

Видим, что ранжирование по близости эмбеддингов работает неплохо, но не идеально.

С одной стороны, документы про Юпитер действительно поднялись выше, что документы про водород. С другой стороны, на первое место вылез документы, содержащий все ключевые слова, но не отвечающий на вопрос пользователя.

Посмотрим теперь как ранжируются документы по второму запросу про водород:

In [ ]:
print(f"Query: {queries[1]}")
rank_docs(docs, scores[1])

Тут у нас аналогичная картина -- ранжирование хорошее, но не идеальное.

## Ранжирование

Попробуем немного исправить ситуацию и улучшить качество ранжирования за счет использования моделей кросс-энкодеров (реранкеров).

На всякий случай вспомним:
- модели-эмбеддеры (их еще называют _би-энкодеры_) получают на вход по отдельности текст документа или запроса, и выдают эмбеддинги
- кросс-энкодеры получают на вход сразу два текста: запроса и документа, и выдают число (ранк), по которому можно ранжировать документы

Главный минусы кросс-энкодеров:
- их надо применять ко всем парам запрос-документ
- эмбеддинги документов невозможно предрасчитать заранее и сохранить в векторном индексе

Но есть и плюс: ранжирование получается гораздо более точным.

Загрузим модели, тут тоже будем использовать легкую модель _MiniLM_:

In [ ]:
# Which model?
ce_model_name = "cross-encoder/ms-marco-MiniLM-L6-v2"
#ce_model_name = "Qwen/Qwen3-Reranker-0.6B"

# Load model
ce_model = sbert.CrossEncoder(ce_model_name, device=str(device), model_kwargs=model_kwargs)
logging.info(f"loaded cross-encoder model: ce_model_name = {ce_model_name}")

Представим данные в виде пар запрос-документ, т.к. в этом формате их ожидает модель:

In [ ]:
num_docs = len(docs)

# Create query/doc pairs
q0 = [queries[0]] * num_docs
pairs0 = list(zip(q0, docs))
pairs0

Аналогично для второго запроса:

In [ ]:
q1 = [queries[1]] * num_docs
pairs1 = list(zip(q1, docs))
pairs1

Применим наши кросс-энкодеры к парам запрос-документ:

In [ ]:
start = timer()
scores0 = ce_model.predict(pairs0)
print(scores0)
logging.info(f"reranked: query = {queries[0]} elapsed = {timer() - start:.3f}")

Аналогично и для второго запроса:

In [ ]:
start = timer()
scores1 = ce_model.predict(pairs1)
print(scores1)
logging.info(f"reranked: query = {queries[1]} elapsed = {timer() - start:.3f}")

Посмотрим что получилось, сначала для первого запроса про Юпитер:

In [ ]:
print(f"Query: {queries[0]}")
rank_docs(docs, scores0)

Результат стал гораздо лучше!

Теперь документы не просто разделились на релевантные и совсем не релевантные, но и внутри группы релевантных более релевантные оказались выше, чем менее релевантные.

И, аналогично, для второго:

In [ ]:
print(f"Query: {queries[1]}")
rank_docs(docs, scores1)

Обратим внимание на значения ранков -- это больше не косинусное расстояние, а некая величина, не имеющая физического смысла.

Распространенная ошибка: подбирать какой-то порог на значение ранка, и фильровать по нему документы -- так делать нельзя!

Таким образом, мы научились:
- пользоваться библиотекой _sentence_transformers_
- эмбеддить запросы и документы
- считать скоры семантической близости
- переранжировать документы с помощью моделей кросс-энкодеров

Дальше для самостоятельного изучения рекомендуем:
- попробовать другие модели
- взять датасет побольше, напр. полный _MS MARCO_
- научиться считать метрики качества ранжирования и сравнить разные модели между собой